In [ ]:
import pandas as pd

print("กำลังโหลดและทำความสะอาดข้อมูลผู้ป่วย...")

# 1. โหลดข้อมูลจากไฟล์ Excel ของคุณ (แก้เป็น read_excel แล้ว)
df = pd.read_excel(r"D:\Project_Dengue\DHF\Final_Provinces_Fixed\Surin.xlsx")

# 2. แปลงปี พ.ศ. เป็น ค.ศ.
df['Year_CE'] = df['Year'] - 543

# 3. ลบคอลัมน์ยอดรวมทิ้ง เพราะเราจะดูแค่รายเดือน
df = df.drop(columns=['Total_C', 'Total_D'])

# 4. แปลงร่างตารางจาก "แนวนอน" เป็น "แนวตั้ง" (Melt)
df_melted = df.melt(id_vars=['Year', 'Year_CE'], var_name='Month_Type', value_name='Count')

# 5. แยกคำ เช่น 'Jan_C' ออกเป็น 2 คอลัมน์ (Month กับ Type)
df_melted[['Month', 'Type']] = df_melted['Month_Type'].str.split('_', expand=True)

# 6. จับ 'C' (Cases/ผู้ป่วย) และ 'D' (Deaths/ผู้เสียชีวิต) แยกเป็น 2 คอลัมน์คู่กัน
df_pivot = df_melted.pivot_table(index=['Year', 'Year_CE', 'Month'], columns='Type', values='Count').reset_index()
df_pivot = df_pivot.rename(columns={'C': 'Cases', 'D': 'Deaths'})

# 7. แปลงชื่อเดือนภาษาอังกฤษ ให้เป็นตัวเลข
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
df_pivot['Month_Num'] = df_pivot['Month'].map(month_map)

# 8. จัดเรียงข้อมูลตาม ปี และ เดือน ให้ถูกต้องตามลำดับเวลา
df_pivot = df_pivot.sort_values(by=['Year_CE', 'Month_Num']).reset_index(drop=True)

# 9. สร้างคอลัมน์ Date (YYYY-MM-01) เตรียมไว้
df_pivot['Date'] = pd.to_datetime(df_pivot['Year_CE'].astype(str) + '-' + df_pivot['Month_Num'].astype(str) + '-01')

# 10. เลือกเฉพาะคอลัมน์ที่สวยงามและจำเป็น
df_clean = df_pivot[['Date', 'Year', 'Year_CE', 'Month', 'Cases', 'Deaths']]

# 11. เซฟออกมาเป็นไฟล์ใหม่
output_file = r'D:\Project_Dengue\DHF\DHF_clean\Surin.csv'
df_clean.to_csv(output_file, index=False)

print(f"จัดระเบียบเสร็จแล้ว! บันทึกไฟล์ใหม่ชื่อ: {output_file}")
print("\nตัวอย่างหน้าตาข้อมูล 5 บรรทัดแรกที่ได้:")
print(df_clean.head())

⏳ กำลังโหลดและทำความสะอาดข้อมูลผู้ป่วย...
✅ จัดระเบียบเสร็จแล้ว! บันทึกไฟล์ใหม่ชื่อ: D:\Project_Dengue\DHF\DHF_clean\Surin.csv

ตัวอย่างหน้าตาข้อมูล 5 บรรทัดแรกที่ได้:
Type       Date  Year  Year_CE Month  Cases  Deaths
0    2003-01-01  2546     2003   Jan    1.0    19.0
1    2003-02-01  2546     2003   Feb    0.0    35.0
2    2003-03-01  2546     2003   Mar    0.0    33.0
3    2003-04-01  2546     2003   Apr    0.0    79.0
4    2003-05-01  2546     2003   May    1.0     0.0
